# CIC-IDS2017 source and cleaning audit

This notebook is for auditing local source CSV files. It does not download the dataset. Place the eight CIC-IDS2017 CSV files in `data/raw/` and run the cells from the repository root. The source used in the reported experiment had a substantial missing-label/missing-value problem in the Thursday Web Attacks file; inspect counts before accepting the cleaned dataset as definitive.

**Memory note:** full DataFrame concatenation can require several gigabytes of RAM. Run on a machine with sufficient memory, or adapt the audit to read CSVs in chunks and report each transformation explicitly.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RAW_DIR = Path("data/raw")
SOURCE_FILES = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]
missing = [name for name in SOURCE_FILES if not (RAW_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        "Missing local source CSVs in data/raw/:\n" + "\n".join(missing)
    )
print(f"Found {len(SOURCE_FILES)} expected source files under {RAW_DIR}")

In [ ]:
frames = []
file_rows = []
for name in SOURCE_FILES:
    part = pd.read_csv(RAW_DIR / name, low_memory=False)
    part.columns = [str(c).strip() for c in part.columns]
    label_candidates = [c for c in part.columns if c.lower() == "label"]
    if not label_candidates:
        raise ValueError(f"No Label column found in {name}")
    label = label_candidates[0]
    x = part.select_dtypes(include=[np.number])
    infinity_rows = np.isinf(x.to_numpy(dtype="float64", copy=False)).any(axis=1) if len(x.columns) else np.zeros(len(part), dtype=bool)
    file_rows.append({
        "source_file": name,
        "raw_rows": len(part),
        "missing_label_rows": int(part[label].isna().sum()),
        "rows_any_missing": int(part.isna().any(axis=1).sum()),
        "rows_any_infinite": int(infinity_rows.sum()),
    })
    frames.append(part)
audit = pd.DataFrame(file_rows)
display(audit)

In [ ]:
raw = pd.concat(frames, ignore_index=True)
label_col = next(c for c in raw.columns if c.strip().lower() == "label")
print("Merged shape:", raw.shape)
print("Merged label counts (including missing labels):")
display(raw[label_col].astype("string").value_counts(dropna=False).rename_axis("label").to_frame("count").head(30))

# Match the broad cleaning rule documented for the reported experiment.
numeric_cols = raw.select_dtypes(include=[np.number]).columns
raw[numeric_cols] = raw[numeric_cols].replace([np.inf, -np.inf], np.nan)
before = len(raw)
clean = raw.dropna().copy()
print(f"Rows removed for any missing/non-finite value: {before - len(clean):,}")
print("After missing/non-finite cleanup:", clean.shape)

constant_cols = [c for c in clean.columns if c != label_col and clean[c].nunique(dropna=False) <= 1]
clean = clean.drop(columns=constant_cols)
print("Constant non-label columns removed:", len(constant_cols))
before_dedup = len(clean)
clean = clean.drop_duplicates().reset_index(drop=True)
print(f"Duplicate rows removed: {before_dedup - len(clean):,}")
print("Final cleaned shape:", clean.shape)

In [ ]:
# Identify model feature columns conservatively: remove ID/time fields and retain numeric columns only.
excluded_terms = {"flow id", "source ip", "destination ip", "timestamp"}
excluded = [c for c in clean.columns if c.strip().lower() in excluded_terms]
model_frame = clean.drop(columns=excluded, errors="ignore")
label_col = next(c for c in model_frame.columns if c.strip().lower() == "label")
features = model_frame.drop(columns=[label_col]).select_dtypes(include=[np.number])
y = model_frame[label_col].astype(str).str.strip().str.lower().ne("benign").astype("int8")
print("Excluded identifier/time fields:", excluded)
print("Numeric model feature count:", features.shape[1])
print("Label counts (0=Benign, 1=Attack):")
display(y.map({0:"Benign", 1:"Attack"}).value_counts().rename_axis("label").to_frame("count"))

## Audit interpretation

The reported run recorded 3,119,345 merged rows before cleaning, 2,827,876 rows after removing any row with missing or infinite values, and 2,827,677 rows after removing 199 duplicates. The final model input had 72 numeric features after dropping constant columns and excluding four identifier/time fields.

For the Thursday Web Attacks file specifically, the audit recorded 288,602 missing-label rows, 288,622 rows with any missing values, 135 rows with an infinite numeric value, 20 rows overlapping the missing-value and infinite-value sets, 288,737 rows removed by the union rule, and 170,231 retained rows.

If your local CSVs yield different counts, stop and investigate file versions, headers, encodings, and cleaning rules. Do not modify reported result CSVs to make them agree with a different run; record the discrepancy and repeat the evaluation transparently.